# Mini-TP 6 — Tu modelo en el Data Lake (individual)

**Operaciones de Aprendizaje Automático II · CEIA · FIUBA**

## Consigna
Subir el artefacto de **un modelo** a un Data Lake (MinIO/S3) y **servirlo cargándolo desde allí**, no desde disco. El bucket se organiza con **zonas** y **versión**.

### Qué se entrega
1. El bucket con zonas (`raw`, `curated`, `models`) y el modelo en `models/v1/…`.
2. Una función que **carga el modelo desde el lake** y predice.
3. (Opcional) El experimento registrado con **MLflow apuntando a MinIO/S3**.
4. Una reflexión breve: ¿por qué servir desde el lake y no hornear el modelo en la imagen?

> Requisitos: MinIO levantado con Docker (ver el README de la clase 6) y las dependencias: `uv add boto3 pandas pyarrow scikit-learn mlflow joblib`.

## 0. Configuración
Credenciales por variables de entorno y endpoint de MinIO. El notebook **falla de forma explícita** si MinIO no está disponible: no hay modo local de respaldo, porque servir el modelo desde disco invalidaría el ejercicio.

In [8]:
import os
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"
BUCKET = "datalake-tp6"
MODEL_VERSION = "v1"          # versión activa del modelo en la zona models/
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET, "| Versión:", MODEL_VERSION)

Endpoint: http://localhost:9000 | Bucket: datalake-tp6 | Versión: v1


In [9]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ["AWS_DEFAULT_REGION"],
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"},
                  connect_timeout=5, retries={"max_attempts": 1}),
)
try:
    s3.list_buckets()
except Exception as exc:
    raise RuntimeError(
        f"MinIO no está disponible en {ENDPOINT}. Levantalo con Docker (ver README de la clase 6) "
        f"y volvé a ejecutar. Error original: {exc}"
    ) from exc
print("Cliente S3 listo.")

Cliente S3 listo.


## 1. Crear el bucket y las zonas

In [10]:
ZONAS = ["raw", "curated", "models"]

def crear_bucket(bucket_name):
    existentes = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
    if bucket_name not in existentes:
        s3.create_bucket(Bucket=bucket_name)
    for zona in ZONAS:
        s3.put_object(Bucket=bucket_name, Key=f"{zona}/.keep", Body=b"")
    print(f"Bucket '{bucket_name}' y zonas {ZONAS} listas.")

def listar_lake(bucket, prefix=""):
    """Imprime los objetos del bucket (sin los marcadores .keep)."""
    paginas = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix)
    for pag in paginas:
        for o in pag.get("Contents", []):
            if not o["Key"].endswith(".keep"):
                print(f"  {o['Key']:<50} {o['Size']:>9,} bytes")

crear_bucket(BUCKET)

Bucket 'datalake-tp6' y zonas ['raw', 'curated', 'models'] listas.


## 2. Entrenar el modelo
Un modelo simple de scikit-learn sobre `digits`; el flujo es el mismo para cualquier modelo serializable.

In [11]:
import io
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

modelo = LogisticRegression(max_iter=500).fit(Xtr, ytr)
acc_local = float((modelo.predict(Xte) == yte).mean())
print(f"Modelo: {type(modelo).__name__}, accuracy local = {acc_local:.3f}")

df_raw = pd.DataFrame(Xtr, columns=[f"f{i}" for i in range(Xtr.shape[1])])
df_raw["label"] = ytr
df_curated = df_raw.copy()

Modelo: LogisticRegression, accuracy local = 0.971


## 3. Subir dataset y modelo al lake
- `raw/`: dato tal cual (CSV).
- `curated/`: dato listo para consumo (Parquet).
- `models/<versión>/`: artefacto del modelo + `metadata.json` con su accuracy y versión de librerías.

In [12]:
import json, sklearn

dataset_name = "digits"
raw_key = f"raw/{dataset_name}/train.csv"
curated_key = f"curated/{dataset_name}/train.parquet"
model_key = f"models/{MODEL_VERSION}/logreg_digits.pkl"
meta_key = f"models/{MODEL_VERSION}/metadata.json"

s3.put_object(Bucket=BUCKET, Key=raw_key, Body=df_raw.to_csv(index=False).encode("utf-8"))

buf = io.BytesIO()
df_curated.to_parquet(buf, index=False)
s3.put_object(Bucket=BUCKET, Key=curated_key, Body=buf.getvalue())

buf = io.BytesIO()
joblib.dump(modelo, buf)
s3.put_object(Bucket=BUCKET, Key=model_key, Body=buf.getvalue())

metadata = {"version": MODEL_VERSION, "model": type(modelo).__name__, "accuracy": acc_local,
            "sklearn": sklearn.__version__, "dataset": raw_key}
s3.put_object(Bucket=BUCKET, Key=meta_key, Body=json.dumps(metadata, indent=2).encode("utf-8"))

print("Contenido del lake:")
listar_lake(BUCKET)

Contenido del lake:
  curated/digits/train.parquet                          82,429 bytes
  models/v1/logreg_digits.pkl                            6,119 bytes
  models/v1/metadata.json                                  147 bytes
  raw/digits/train.csv                               1,652,325 bytes


## 4. Servir el modelo desde el lake
El corazón del Mini-TP: la función **descarga el artefacto del lake** (no lo lee de disco ni reutiliza el objeto en memoria) y predice.

In [13]:
def cargar_modelo_desde_lake(bucket, key):
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return joblib.load(io.BytesIO(body))

def predecir(entradas, version=MODEL_VERSION):
    """Carga la versión indicada desde el lake y devuelve las clases predichas."""
    m = cargar_modelo_desde_lake(BUCKET, f"models/{version}/logreg_digits.pkl")
    return m.predict(entradas)

modelo_lake = cargar_modelo_desde_lake(BUCKET, model_key)
assert modelo_lake is not modelo, "debe ser una copia deserializada desde el lake"

pred = predecir(Xte[:5])
print("Predicciones desde el lake:", pred.tolist())
print("Etiquetas reales          :", yte[:5].tolist())

# El modelo servido desde el lake se comporta igual que el entrenado.
assert np.array_equal(modelo_lake.predict(Xte), modelo.predict(Xte))
acc_lake = float((predecir(Xte) == yte).mean())
print(f"Accuracy del modelo servido desde el lake: {acc_lake:.3f} (local: {acc_local:.3f})")

Predicciones desde el lake: [2, 0, 4, 9, 4]
Etiquetas reales          : [2, 0, 4, 9, 4]
Accuracy del modelo servido desde el lake: 0.971 (local: 0.971)


## 5. (Opcional) MLflow apuntando a MinIO/S3
El *tracking* (parámetros y métricas) va a SQLite local; los **artefactos** se guardan en `s3://<bucket>/mlflow` (MinIO).

In [14]:
import logging
import mlflow

for nombre in ("alembic", "mlflow"):
    logging.getLogger(nombre).setLevel(logging.WARNING)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT
mlflow.set_tracking_uri("sqlite:///mlflow_tp6.db")

EXPERIMENTO = "tp6-datalake"
if mlflow.get_experiment_by_name(EXPERIMENTO) is None:
    mlflow.create_experiment(EXPERIMENTO, artifact_location=f"s3://{BUCKET}/mlflow")
mlflow.set_experiment(EXPERIMENTO)

with mlflow.start_run(run_name="digits-model") as run:
    mlflow.log_param("model_type", type(modelo).__name__)
    mlflow.log_param("model_version", MODEL_VERSION)
    mlflow.log_metric("accuracy", acc_local)
    joblib.dump(modelo, "modelo_tp6_local.pkl")
    mlflow.log_artifact("modelo_tp6_local.pkl", artifact_path="model")
    print("Run:", run.info.run_id)
    print("artifact_uri:", run.info.artifact_uri)

print("\nArtefactos de MLflow dentro del lake:")
listar_lake(BUCKET, prefix="mlflow/")

c:\Users\SofiJos\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\_internal\_fields.py:160: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Run: de96c84d631045ef9692a4f13030ca41
artifact_uri: s3://datalake-tp6/mlflow/de96c84d631045ef9692a4f13030ca41/artifacts

Artefactos de MLflow dentro del lake:
  mlflow/de96c84d631045ef9692a4f13030ca41/artifacts/model/modelo_tp6_local.pkl     6,119 bytes


## 6. Reflexión

**1. ¿Por qué servir desde el lake y no hornear el modelo en la imagen?**
- **Desacopla ciclo de vida del modelo y del código.** Con el modelo dentro de la imagen, cada reentrenamiento obliga a rebuild, push y redeploy del servicio. Desde el lake, basta con subir `models/v2/…` y apuntar el servicio a esa versión.
- **Un mismo artefacto para todos los entornos.** Desarrollo, staging y producción cargan exactamente el mismo objeto desde el mismo lugar, sin copias que se desincronicen entre imágenes.
- **Versionado, auditoría y rollback.** Las versiones viven lado a lado (`v1`, `v2`) con su `metadata.json` (accuracy, versión de librerías, dataset de origen). Volver atrás es cambiar `MODEL_VERSION`, no reconstruir una imagen vieja.
- **Imágenes más livianas y reutilizables.** La imagen contiene solo el código de serving; el peso del modelo (aquí 6 KB, pero en la práctica cientos de MB o GB) no se multiplica por cada build.
- **Costos y contrapartidas.** El servicio depende de la disponibilidad del lake y paga latencia al arrancar (descarga y deserialización); se mitiga cargando el modelo una sola vez al iniciar y cacheándolo. Además hay que controlar el acceso al bucket (credenciales y permisos de solo lectura para el servicio) y fijar versiones de librerías, porque un `.pkl` de scikit-learn solo es compatible con la versión que lo generó; por eso la versión de `sklearn` queda registrada en la metadata.

**2. Versionado.** El artefacto se guarda como `models/v1/…` y el siguiente iría en `models/v2/…`. El servicio elige la versión activa por configuración (`MODEL_VERSION`) y deja la anterior disponible para rollback.

**3. Predicciones servidas por GraphQL.** Las guardaría en `curated/predictions/` (formato Parquet, particionado por fecha), porque son tablas analíticas ya procesadas y listas para consumo. `raw` quedaría para datos de entrada sin transformar, y cada predicción incluiría la versión del modelo que la produjo para mantener la trazabilidad.
